# PutStrike iTransformer Training Notebook

**iTransformer: Inverted Transformers Are Effective for Time Series Forecasting** (ICLR 2024, Liu et al.)

## Architecture Overview

The iTransformer **inverts** the standard Transformer design:
- **Standard Transformer**: each **time step** is a token, attention captures temporal patterns
- **iTransformer**: each **feature/variate** is a token, attention captures cross-variate correlations

### Why iTransformer for stock prediction?
1. **Cross-variate attention** — captures how features interact (e.g., VIX spike + put/call ratio surge together)
2. **Per-variate temporal embedding** — each feature's lookback window is projected independently
3. **Scales with high-dimensional features** — 80+ features, each treated as a token
4. **Shared output projection** — single `Linear(d_model → horizon)` applied per variate (original paper design)

### Key Design Decisions (Research-Validated)
- **Walk-forward validation** instead of simple train/val split (prevents look-ahead bias)
- **Shared output projection** per the original paper — NOT per-variate heads or attention pooling
- **Instance normalization (RevIN)** — per-window mean/std normalization, reversed on output
- **Directional accuracy** as primary metric (more important than MSE for trading)
- **Huber loss** instead of MSE (robust to outlier returns from earnings/events)
- **Learning rate warmup + cosine decay** (standard for Transformers)
- **Shuffle windowed samples** across batches (temporal order preserved within each window)

### Setup
1. Run in Google Colab with **GPU runtime** (Runtime > Change runtime type > T4 GPU)
2. Execute cells in order
3. Copy ngrok URL and paste into PutStrike settings

In [ ]:
# Cell 1: Install Dependencies
import subprocess
import sys

packages = [
    "torch", "numpy", "pandas", "yfinance",
    "flask", "pyngrok", "scikit-learn",
    "matplotlib",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("[OK] All dependencies installed.")

In [ ]:
# Cell 2: Configuration
import os
import torch
import numpy as np

# ── NGROK AUTH TOKEN ──
# Get your free token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "YOUR_NGROK_TOKEN_HERE")

# ── Training Configuration ──
SYMBOLS = [
    # High-liquidity stocks matching PutStrike's screener
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA",
    "JPM", "V", "JNJ", "PG", "XOM", "UNH",
    "SPY", "QQQ", "IWM",
]

# Architecture hyperparameters (validated against iTransformer paper)
LOOKBACK_WINDOW = 60      # 60 trading days (~3 months) of history per sample
FORECAST_HORIZON = 30     # Predict 30 days ahead (matches put selling DTE)
D_MODEL = 128             # Transformer hidden dimension (reduced from 256 — prevents overfitting on financial data)
N_HEADS = 8               # Attention heads
N_LAYERS = 3              # Transformer layers (increased from 2 for deeper cross-variate learning)
D_FF = 256                # Feed-forward dimension
DROPOUT = 0.2             # Higher dropout for noisy financial data (was 0.1)

# Training hyperparameters
BATCH_SIZE = 64           # Larger batch for stable gradients
EPOCHS = 80               # More epochs with early stopping
LEARNING_RATE = 3e-4      # Standard for Transformers (Vaswani et al.)
WARMUP_EPOCHS = 5         # LR warmup (critical for Transformers)
WEIGHT_DECAY = 1e-3       # Stronger regularization for financial data
PATIENCE = 15             # Early stopping patience
TRAIN_SPLIT = 0.7         # 70% train, 15% val, 15% test (walk-forward)
VAL_SPLIT = 0.85          # Of remaining 30%, split into val/test

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[CONFIG] {len(SYMBOLS)} symbols, lookback={LOOKBACK_WINDOW}, horizon={FORECAST_HORIZON}")
print(f"[CONFIG] d_model={D_MODEL}, layers={N_LAYERS}, heads={N_HEADS}, dropout={DROPOUT}")
print(f"[DEVICE] {device}" + (f" — {torch.cuda.get_device_name(0)}" if device.type == 'cuda' else ""))

In [ ]:
# Cell 3: Feature Engineering
# Mirrors PutStrike's TypeScript features.ts — computes 80+ technical/statistical features from OHLCV

import pandas as pd
import yfinance as yf
from typing import Dict, List, Optional, Tuple

def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute 80+ features from OHLCV data.
    Mirrors the TypeScript feature engineering in src/lib/features.ts.
    """
    feat = pd.DataFrame(index=df.index)
    close = df["Close"].squeeze()
    high = df["High"].squeeze()
    low = df["Low"].squeeze()
    volume = df["Volume"].squeeze()
    open_ = df["Open"].squeeze()

    # ── Moving Averages (10 features) ──
    for p in [5, 10, 20, 50, 200]:
        sma = close.rolling(p).mean()
        feat[f"price_vs_sma_{p}_pct"] = ((close - sma) / sma) * 100

    for p in [5, 12, 26]:
        ema_val = close.ewm(span=p, adjust=False).mean()
        feat[f"price_vs_ema_{p}_pct"] = ((close - ema_val) / ema_val) * 100

    feat["sma_20_50_cross"] = (
        close.rolling(20).mean() > close.rolling(50).mean()
    ).astype(float)

    feat["sma_50_200_cross"] = (
        close.rolling(50).mean() > close.rolling(200).mean()
    ).astype(float)

    # ── RSI (3 features) ──
    for p in [7, 14, 21]:
        delta = close.diff()
        gain = delta.where(delta > 0, 0).rolling(p).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(p).mean()
        rs = gain / (loss + 1e-10)
        feat[f"rsi_{p}"] = 100 - 100 / (1 + rs)

    # ── MACD (4 features) ──
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    macd_signal = macd_line.ewm(span=9, adjust=False).mean()
    feat["macd_histogram"] = (macd_line - macd_signal) / close * 100  # Normalize by price
    feat["macd_cross_above"] = (
        (macd_line > macd_signal) & (macd_line.shift(1) <= macd_signal.shift(1))
    ).astype(float)

    # ── Bollinger Bands (3 features) ──
    bb_sma = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    feat["bb_width"] = (4 * bb_std / bb_sma) * 100
    feat["bb_pctb"] = (close - (bb_sma - 2 * bb_std)) / (4 * bb_std + 1e-10)

    # ── ATR (4 features) ──
    for p in [7, 14]:
        tr1 = high - low
        tr2 = (high - close.shift(1)).abs()
        tr3 = (low - close.shift(1)).abs()
        # Use np.maximum for element-wise max of Series (avoids pd.concat MultiIndex issues)
        tr = np.maximum(np.maximum(tr1, tr2), tr3)
        atr = tr.rolling(p).mean()
        feat[f"atr_{p}_pct"] = (atr / close) * 100

    # ── Volume (6 features) ──
    feat["volume_ratio_5_20"] = volume.rolling(5).mean() / (volume.rolling(20).mean() + 1)
    feat["relative_volume"] = volume / (volume.rolling(20).mean() + 1)
    # OBV trend (normalized)
    obv = (np.sign(close.diff()) * volume).cumsum()
    obv_norm = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-10)
    feat["obv_zscore"] = obv_norm
    # CMF
    clv = ((close - low) - (high - close)) / (high - low + 1e-10)
    feat["cmf_20"] = (clv * volume).rolling(20).sum() / (volume.rolling(20).sum() + 1)
    feat["volume_zscore_20"] = (
        (volume - volume.rolling(20).mean()) / (volume.rolling(20).std() + 1e-10)
    )

    # ── Stochastic (2 features) ──
    low14 = low.rolling(14).min()
    high14 = high.rolling(14).max()
    feat["stoch_k"] = ((close - low14) / (high14 - low14 + 1e-10)) * 100
    feat["stoch_d"] = feat["stoch_k"].rolling(3).mean()

    # ── Williams %R ──
    feat["williams_r"] = ((high14 - close) / (high14 - low14 + 1e-10)) * -100

    # ── ROC (3 features) ──
    for p in [5, 10, 20]:
        feat[f"roc_{p}"] = close.pct_change(p) * 100

    # ── CCI ──
    tp = (high + low + close) / 3
    tp_sma = tp.rolling(20).mean()
    tp_mad = tp.rolling(20).apply(lambda x: np.mean(np.abs(x - np.mean(x))))
    feat["cci_20"] = (tp - tp_sma) / (0.015 * tp_mad + 1e-10)

    # ── Aroon (3 features) ──
    feat["aroon_up"] = high.rolling(25).apply(lambda x: x.argmax() / 24 * 100)
    feat["aroon_down"] = low.rolling(25).apply(lambda x: x.argmin() / 24 * 100)
    feat["aroon_oscillator"] = feat["aroon_up"] - feat["aroon_down"]

    # ── Returns (5 features) ──
    for p in [1, 5, 10, 20, 60]:
        feat[f"return_{p}d"] = close.pct_change(p)

    # ── Volatility (4 features) ──
    log_ret = np.log(close / close.shift(1))
    for p in [5, 10, 20, 60]:
        feat[f"volatility_{p}d"] = log_ret.rolling(p).std() * np.sqrt(252)

    # ── Higher Moments (4 features) ──
    feat["skewness_20d"] = log_ret.rolling(20).skew()
    feat["skewness_60d"] = log_ret.rolling(60).skew()
    feat["kurtosis_20d"] = log_ret.rolling(20).kurt()
    feat["kurtosis_60d"] = log_ret.rolling(60).kurt()

    # ── Autocorrelation (3 features) ──
    for lag in [1, 3, 5]:
        feat[f"autocorr_lag_{lag}"] = log_ret.rolling(30).apply(
            lambda x: x.autocorr(lag) if len(x) >= lag + 2 else 0
        )

    # ── Z-Scores (4 features) ──
    for p in [20, 50, 100, 200]:
        roll_mean = close.rolling(p).mean()
        roll_std = close.rolling(p).std()
        feat[f"zscore_{p}"] = (close - roll_mean) / (roll_std + 1e-10)

    # ── Percentile Ranks (3 features) ──
    for p in [20, 60, 252]:
        feat[f"percentile_rank_{p}d"] = close.rolling(p).apply(
            lambda x: (x < x.iloc[-1]).sum() / len(x) * 100 if len(x) == p else 50
        )

    # ── Max Drawdown (2 features) ──
    for p in [20, 60]:
        rolling_max = close.rolling(p).max()
        feat[f"max_drawdown_{p}d"] = (close - rolling_max) / (rolling_max + 1e-10)

    # ── Up/Down Ratios (2 features) ──
    for p in [10, 20]:
        feat[f"up_ratio_{p}d"] = (close.diff() > 0).rolling(p).mean()

    # ── Gap Features (2 features) ──
    gap = (open_ - close.shift(1)) / (close.shift(1) + 1e-10)
    feat["avg_gap_20d"] = gap.rolling(20).mean()
    feat["gap_frequency_20d"] = (gap.abs() > 0.01).rolling(20).mean()

    # ── Calendar Features (5 features) ──
    dates = pd.to_datetime(df.index)
    feat["day_of_week"] = dates.dayofweek / 4  # Normalized 0-1
    feat["month_sin"] = np.sin(2 * np.pi * dates.month / 12)
    feat["month_cos"] = np.cos(2 * np.pi * dates.month / 12)
    feat["is_quarter_end"] = dates.month.isin([3, 6, 9, 12]).astype(float)
    day_of_month = dates.day
    feat["is_opex_week"] = ((day_of_month >= 15) & (day_of_month <= 21)).astype(float)

    # ── Trend Strength (3 features) ──
    feat["price_slope_20"] = close.rolling(20).apply(
        lambda x: np.polyfit(range(len(x)), x / x.iloc[0], 1)[0] if len(x) == 20 else 0  # Normalized slope
    )
    feat["price_slope_50"] = close.rolling(50).apply(
        lambda x: np.polyfit(range(len(x)), x / x.iloc[0], 1)[0] if len(x) == 50 else 0
    )
    # Ichimoku
    tenkan = (high.rolling(9).max() + low.rolling(9).min()) / 2
    kijun = (high.rolling(26).max() + low.rolling(26).min()) / 2
    feat["ichimoku_tk_cross"] = (tenkan > kijun).astype(float)

    # ── Volatility Regime (2 features) ──
    feat["vol_regime_ratio"] = feat["volatility_20d"] / (feat["volatility_60d"] + 1e-10)
    feat["vol_expanding"] = (feat["volatility_20d"] > feat["volatility_60d"]).astype(float)

    # Clean up
    feat = feat.replace([np.inf, -np.inf], np.nan)
    feat = feat.fillna(0)

    return feat


print(f"[OK] Feature engineering function defined ({compute_features.__name__})")

In [ ]:
# Cell 4: Data Download & Preparation

def download_and_prepare_data(
    symbols: List[str],
    lookback: int,
    horizon: int,
) -> Tuple[np.ndarray, np.ndarray, List[str], dict]:
    """
    Download data, compute features, create training samples.
    Uses walk-forward split: data is chronologically ordered per stock.

    Returns:
        X: (num_samples, lookback, num_features)
        y: (num_samples, horizon) — future returns
        feature_names: list of feature column names
        normalization_stats: per-stock mean/std for inference
    """
    all_X = []
    all_y = []
    feature_names = None
    normalization_stats = {}

    for sym in symbols:
        print(f"  Downloading {sym}...", end=" ")
        try:
            df = yf.download(sym, period="5y", interval="1d", progress=False)

            # yfinance >= 0.2.31 returns MultiIndex columns (Price, Ticker)
            # even for single symbols when auto_adjust=True (now the default).
            # Flatten to simple column names so downstream code works.
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            if len(df) < lookback + horizon + 252:  # Need at least 1yr warmup + data
                print(f"skipped (only {len(df)} days)")
                continue

            features = compute_features(df)
            closes = df["Close"].values.flatten()

            if feature_names is None:
                feature_names = list(features.columns)
                print(f"({len(feature_names)} features)")
            else:
                print(f"OK ({len(df)} days)")

            # Z-score normalize per stock (store stats for inference)
            feat_values = features.values
            feat_mean = np.nanmean(feat_values, axis=0, keepdims=True)
            feat_std = np.nanstd(feat_values, axis=0, keepdims=True) + 1e-10
            feat_norm = (feat_values - feat_mean) / feat_std

            normalization_stats[sym] = {
                "mean": feat_mean.flatten().tolist(),
                "std": feat_std.flatten().tolist(),
            }

            # Clip extreme values (robust to outliers)
            feat_norm = np.clip(feat_norm, -5, 5)

            # Create sliding window samples
            for i in range(lookback, len(feat_norm) - horizon):
                X_sample = feat_norm[i - lookback:i]  # (lookback, features)

                # Target: forward returns at each horizon day
                current_price = closes[i]
                future_prices = closes[i + 1:i + horizon + 1]
                if len(future_prices) == horizon and current_price > 0:
                    y_sample = (future_prices - current_price) / current_price
                    all_X.append(X_sample)
                    all_y.append(y_sample)

        except Exception as e:
            print(f"FAILED ({e})")
            continue

    X = np.array(all_X, dtype=np.float32)
    y = np.array(all_y, dtype=np.float32)
    print(f"\n[DATA] {X.shape[0]:,} samples, {X.shape[2]} features, "
          f"lookback={X.shape[1]}, horizon={y.shape[1]}")

    return X, y, feature_names or [], normalization_stats


print("[1/6] Downloading market data and computing features...")
X, y, feature_names, norm_stats = download_and_prepare_data(
    SYMBOLS, LOOKBACK_WINDOW, FORECAST_HORIZON
)
print(f"\n[DATA] Feature names ({len(feature_names)}): {feature_names[:10]}...")

In [ ]:
# Cell 5: iTransformer Model Architecture
# Following the original paper: shared embedding → cross-variate attention → shared projection
# With RevIN (Reversible Instance Normalization) for non-stationary financial data

import torch
import torch.nn as nn
import math


class RevIN(nn.Module):
    """
    Reversible Instance Normalization (Kim et al., ICLR 2022).
    Used in the original iTransformer implementation.
    
    Per-window normalization: subtracts mean and divides by std of each input,
    then reverses this on the output. Critical for non-stationary financial data.
    """
    def __init__(self, num_features: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.affine_weight = nn.Parameter(torch.ones(num_features))
            self.affine_bias = nn.Parameter(torch.zeros(num_features))

    def forward(self, x: torch.Tensor, mode: str = "norm") -> torch.Tensor:
        """
        x: (batch, lookback, num_variates)
        mode: "norm" to normalize, "denorm" to reverse
        """
        if mode == "norm":
            self._mean = x.mean(dim=1, keepdim=True).detach()
            self._std = (x.std(dim=1, keepdim=True) + self.eps).detach()
            x = (x - self._mean) / self._std
            if self.affine:
                x = x * self.affine_weight + self.affine_bias
            return x
        else:  # denorm
            if self.affine:
                x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            # Note: for return prediction we don't denorm (outputs are already returns)
            return x


class iTransformer(nn.Module):
    """
    Inverted Transformer for Time-Series Forecasting (ICLR 2024, Liu et al.).

    Architecture (matching the original paper exactly):
    1. Input: (batch, lookback, num_variates)
    2. RevIN normalization (per-window instance normalization)
    3. INVERT: transpose to (batch, num_variates, lookback)
    4. Shared embedding: Linear(lookback -> d_model) applied to each variate
    5. Add learnable variate tokens
    6. Transformer encoder: self-attention across variates (not time)
    7. Shared projection: Linear(d_model -> forecast_horizon) applied to each variate
    8. Aggregate: weighted combination of per-variate forecasts

    Key: The embedding and projection are single shared Linear layers (not per-variate),
    applied independently to each variate token. This is exactly how the original paper
    and official code (thuml/Time-Series-Library) implement it.
    """

    def __init__(
        self,
        num_variates: int,
        lookback: int,
        forecast_horizon: int,
        d_model: int = 128,
        n_heads: int = 8,
        n_layers: int = 3,
        d_ff: int = 256,
        dropout: float = 0.2,
        use_norm: bool = True,
    ):
        super().__init__()
        self.num_variates = num_variates
        self.lookback = lookback
        self.forecast_horizon = forecast_horizon
        self.d_model = d_model
        self.use_norm = use_norm

        # RevIN for non-stationary data (per the paper's use_norm flag)
        if use_norm:
            self.revin = RevIN(num_variates)

        # Shared embedding: project each variate's lookback → d_model
        # (single Linear applied to all variates — original paper design)
        self.variate_embedding = nn.Linear(lookback, d_model)

        # Learnable variate tokens (variate-level identity encoding)
        self.variate_tokens = nn.Parameter(
            torch.randn(1, num_variates, d_model) * 0.02
        )

        # Transformer encoder layers (attention across variates)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,  # Pre-norm (more stable for deeper models)
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )

        # Shared output projection: d_model → forecast_horizon
        # (single Linear applied to all variates — matches original paper)
        self.projection = nn.Linear(d_model, forecast_horizon, bias=True)

        # Learnable aggregation weights across variates
        # (our addition for single-output forecasting; paper predicts all variates)
        self.agg_weights = nn.Parameter(torch.ones(num_variates) / num_variates)

        # Output refinement
        self.output_head = nn.Sequential(
            nn.LayerNorm(forecast_horizon),
            nn.Linear(forecast_horizon, forecast_horizon),
            nn.Tanh(),  # Bound outputs (returns are typically small)
        )

        self._init_weights()

    def _init_weights(self):
        """Xavier initialization for stable training."""
        for name, p in self.named_parameters():
            if p.dim() > 1 and 'revin' not in name:
                nn.init.xavier_uniform_(p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, lookback, num_variates) — standard time-series format
        Returns:
            forecast: (batch, forecast_horizon) — predicted returns
        """
        # Instance normalization (per-window)
        if self.use_norm:
            x = self.revin(x, mode="norm")

        # INVERT: transpose to (batch, num_variates, lookback)
        x = x.transpose(1, 2)  # (B, V, L)

        # Shared embedding: each variate's lookback → d_model
        tokens = self.variate_embedding(x)  # (B, V, d_model)

        # Add learnable variate identifiers
        tokens = tokens + self.variate_tokens

        # Transformer encoder: attention across variates
        encoded = self.encoder(tokens)  # (B, V, d_model)

        # Shared projection to forecast horizon (applied per-variate)
        per_variate_forecast = self.projection(encoded)  # (B, V, horizon)

        # Weighted aggregation across variates
        weights = torch.softmax(self.agg_weights, dim=0)  # (V,)
        forecast = torch.einsum('bvh,v->bh', per_variate_forecast, weights)  # (B, horizon)

        # Output refinement
        forecast = self.output_head(forecast)

        return forecast

    def get_variate_importance(self) -> np.ndarray:
        """Return softmax-normalized variate aggregation weights."""
        with torch.no_grad():
            return torch.softmax(self.agg_weights, dim=0).cpu().numpy()


# Instantiate and inspect model
model = iTransformer(
    num_variates=X.shape[2],
    lookback=X.shape[1],
    forecast_horizon=y.shape[1],
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    use_norm=True,
).to(device)

param_count = sum(p.numel() for p in model.parameters())
print(f"[MODEL] iTransformer — {param_count:,} parameters")
print(f"  Variates (tokens): {X.shape[2]}")
print(f"  Lookback: {X.shape[1]} days")
print(f"  Forecast: {y.shape[1]} days")
print(f"  d_model: {D_MODEL}, layers: {N_LAYERS}, heads: {N_HEADS}")
print(f"  RevIN: enabled (instance normalization)")

In [ ]:
# Cell 6: Training Loop with Walk-Forward Validation

from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


def train_model(
    X: np.ndarray,
    y: np.ndarray,
    model: nn.Module,
    config: dict,
) -> Tuple[nn.Module, dict]:
    """
    Train the iTransformer model with walk-forward validation.

    Walk-forward split (no data leakage):
    - Train: first 70% of samples (chronological)
    - Validation: next 15%
    - Test: final 15% (held out, evaluated after training)

    Key design decisions (research-validated):
    - Huber loss: robust to outlier returns from earnings/black swan events
    - LR warmup + cosine decay: standard for Transformers
    - shuffle=True for DataLoader: shuffling windowed samples across batches is
      correct (temporal order preserved WITHIN each window). Confirmed by the
      official iTransformer implementation, HuggingFace TST, and research.
    - Walk-forward split: train/val/test boundaries respect chronological order
    - Directional accuracy: primary metric for trading (more useful than MSE)
    """
    n = X.shape[0]
    train_end = int(n * config["train_split"])
    val_end = int(n * config["val_split"])

    X_train, y_train = X[:train_end], y[:train_end]
    X_val, y_val = X[train_end:val_end], y[train_end:val_end]
    X_test, y_test = X[val_end:], y[val_end:]

    print(f"  Train: {len(X_train):,} samples")
    print(f"  Val:   {len(X_val):,} samples")
    print(f"  Test:  {len(X_test):,} samples")

    # DataLoaders
    # shuffle=True for training: shuffles windowed samples across batches
    # (temporal order within each 60-day window is preserved — this is correct
    #  and standard practice per iTransformer official code and literature)
    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    val_ds = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val))
    test_ds = TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test))

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"])
    test_loader = DataLoader(test_ds, batch_size=config["batch_size"])

    # Optimizer with weight decay
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
        betas=(0.9, 0.999),
    )

    # LR scheduler: warmup + cosine decay
    def lr_lambda(epoch):
        if epoch < config["warmup_epochs"]:
            return (epoch + 1) / config["warmup_epochs"]
        progress = (epoch - config["warmup_epochs"]) / max(1, config["epochs"] - config["warmup_epochs"])
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # Huber loss — robust to outlier returns (earnings gaps, black swans)
    criterion = nn.HuberLoss(delta=0.02)  # delta=2% return

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "val_dir_acc": [], "lr": []}

    for epoch in range(config["epochs"]):
        # ── Training ──
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # ── Validation ──
        model.eval()
        val_loss = 0
        all_pred = []
        all_true = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pred = model(X_batch)
                val_loss += criterion(pred, y_batch).item()
                all_pred.append(pred.cpu())
                all_true.append(y_batch.cpu())
        val_loss /= len(val_loader)

        # Directional accuracy (30-day endpoint)
        preds = torch.cat(all_pred)
        trues = torch.cat(all_true)
        dir_acc = ((preds[:, -1] > 0) == (trues[:, -1] > 0)).float().mean().item() * 100

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_dir_acc"].append(dir_acc)
        history["lr"].append(current_lr)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            marker = "*best*" if patience_counter == 0 else f"(patience {patience_counter}/{config['patience']})"
            print(f"  Epoch {epoch+1:3d}/{config['epochs']} — "
                  f"train: {train_loss:.6f}, val: {val_loss:.6f}, "
                  f"dir_acc: {dir_acc:.1f}%, lr: {current_lr:.2e} {marker}")

        if patience_counter >= config["patience"]:
            print(f"  [EARLY STOP] No improvement for {config['patience']} epochs.")
            break

    # Load best model
    if best_state:
        model.load_state_dict(best_state)
    model = model.to(device)

    # ── Test Set Evaluation ──
    model.eval()
    test_preds = []
    test_trues = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            pred = model(X_batch)
            test_preds.append(pred.cpu())
            test_trues.append(y_batch)

    test_preds = torch.cat(test_preds)
    test_trues = torch.cat(test_trues)

    # Metrics
    test_mse = ((test_preds - test_trues) ** 2).mean().item()
    test_mae = (test_preds - test_trues).abs().mean().item()
    test_dir_acc = ((test_preds[:, -1] > 0) == (test_trues[:, -1] > 0)).float().mean().item() * 100

    # Per-horizon directional accuracy
    horizon_dir_acc = []
    for h in range(test_preds.shape[1]):
        acc = ((test_preds[:, h] > 0) == (test_trues[:, h] > 0)).float().mean().item() * 100
        horizon_dir_acc.append(acc)

    history["test_metrics"] = {
        "mse": test_mse,
        "mae": test_mae,
        "dir_acc_30d": test_dir_acc,
        "horizon_dir_acc": horizon_dir_acc,
    }

    print(f"\n[TEST RESULTS]")
    print(f"  MSE:  {test_mse:.6f}")
    print(f"  MAE:  {test_mae:.6f}")
    print(f"  30-Day Directional Accuracy: {test_dir_acc:.1f}%")
    print(f"  7-Day Directional Accuracy:  {horizon_dir_acc[6]:.1f}%")
    print(f"  14-Day Directional Accuracy: {horizon_dir_acc[13]:.1f}%")

    return model, history


print("[2/6] Training iTransformer...")
config = {
    "d_model": D_MODEL, "n_heads": N_HEADS, "n_layers": N_LAYERS,
    "d_ff": D_FF, "dropout": DROPOUT, "batch_size": BATCH_SIZE,
    "epochs": EPOCHS, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS, "patience": PATIENCE,
    "train_split": TRAIN_SPLIT, "val_split": VAL_SPLIT,
}
model, history = train_model(X, y, model, config)

In [ ]:
# Cell 7: Training Visualization & Model Analysis

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("iTransformer Training Results", fontsize=14, fontweight="bold")

# 1. Loss curves
ax1 = axes[0, 0]
ax1.plot(history["train_loss"], label="Train", alpha=0.8)
ax1.plot(history["val_loss"], label="Validation", alpha=0.8)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Huber Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Directional accuracy
ax2 = axes[0, 1]
ax2.plot(history["val_dir_acc"], label="Val Dir Acc (30d)", color="green", alpha=0.8)
ax2.axhline(y=50, color="red", linestyle="--", alpha=0.5, label="Random (50%)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("30-Day Directional Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Per-horizon directional accuracy
ax3 = axes[1, 0]
test_metrics = history["test_metrics"]
days = list(range(1, len(test_metrics["horizon_dir_acc"]) + 1))
ax3.bar(days, test_metrics["horizon_dir_acc"], alpha=0.7, color="steelblue")
ax3.axhline(y=50, color="red", linestyle="--", alpha=0.5, label="Random")
ax3.set_xlabel("Forecast Day")
ax3.set_ylabel("Directional Accuracy (%)")
ax3.set_title("Test: Per-Horizon Directional Accuracy")
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Feature importance (top 20 by aggregation weight)
ax4 = axes[1, 1]
importance = model.get_variate_importance()
top_idx = np.argsort(importance)[-20:]
top_names = [feature_names[i] for i in top_idx]
top_weights = importance[top_idx]
ax4.barh(top_names, top_weights, color="coral", alpha=0.8)
ax4.set_xlabel("Aggregation Weight")
ax4.set_title("Top 20 Feature Importance")
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n[SUMMARY]")
print(f"  Best Val Loss:     {min(history['val_loss']):.6f}")
print(f"  Test MSE:          {test_metrics['mse']:.6f}")
print(f"  Test MAE:          {test_metrics['mae']:.6f}")
print(f"  Test Dir Acc (30d): {test_metrics['dir_acc_30d']:.1f}%")
print(f"  Epochs trained:    {len(history['train_loss'])}")

In [ ]:
# Cell 8: Save Model & Metadata

import json

# Save PyTorch model
torch.save({
    "model_state_dict": model.state_dict(),
    "config": config,
    "feature_names": feature_names,
    "num_features": len(feature_names),
    "lookback": LOOKBACK_WINDOW,
    "horizon": FORECAST_HORIZON,
    "test_metrics": history["test_metrics"],
}, "itransformer_model.pt")

# Save metadata (for the website to display)
metadata = {
    "model_name": "iTransformer",
    "paper": "Inverted Transformers Are Effective for Time Series Forecasting (ICLR 2024)",
    "feature_names": feature_names,
    "num_features": len(feature_names),
    "lookback": LOOKBACK_WINDOW,
    "horizon": FORECAST_HORIZON,
    "architecture": {
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_heads": N_HEADS,
        "d_ff": D_FF,
        "dropout": DROPOUT,
        "parameters": sum(p.numel() for p in model.parameters()),
    },
    "training": {
        "symbols": SYMBOLS,
        "total_samples": X.shape[0],
        "epochs_trained": len(history["train_loss"]),
        "best_val_loss": float(min(history["val_loss"])),
        "loss_function": "HuberLoss(delta=0.02)",
        "optimizer": "AdamW",
        "lr_schedule": "warmup + cosine decay",
        "validation": "walk-forward (70/15/15)",
    },
    "test_metrics": {
        "mse": float(history["test_metrics"]["mse"]),
        "mae": float(history["test_metrics"]["mae"]),
        "directional_accuracy_30d": float(history["test_metrics"]["dir_acc_30d"]),
    },
    "normalization_stats": norm_stats,
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("[3/6] Model saved:")
print(f"  - itransformer_model.pt ({os.path.getsize('itransformer_model.pt') / 1e6:.1f} MB)")
print(f"  - model_metadata.json")

In [ ]:
# Cell 8b: Save Model to Google Drive
# Mount Google Drive and persist the trained model + metadata so it survives
# Colab runtime restarts. Files are saved to /content/drive/MyDrive/PutStrike/

import shutil

try:
    from google.colab import drive
    drive.mount("/content/drive")

    save_dir = "/content/drive/MyDrive/PutStrike"
    os.makedirs(save_dir, exist_ok=True)

    shutil.copy("itransformer_model.pt", os.path.join(save_dir, "itransformer_model.pt"))
    shutil.copy("model_metadata.json", os.path.join(save_dir, "model_metadata.json"))

    model_size = os.path.getsize(os.path.join(save_dir, "itransformer_model.pt")) / 1e6
    print(f"[OK] Model saved to Google Drive:")
    print(f"  - {save_dir}/itransformer_model.pt ({model_size:.1f} MB)")
    print(f"  - {save_dir}/model_metadata.json")
    print(f"\nTo reload later, run:")
    print(f'  checkpoint = torch.load("{save_dir}/itransformer_model.pt")')
except ImportError:
    print("[SKIP] Not running in Google Colab — Google Drive not available.")
    print("  Model saved locally: itransformer_model.pt, model_metadata.json")
except Exception as e:
    print(f"[WARNING] Failed to save to Google Drive: {e}")
    print("  Model is still saved locally: itransformer_model.pt")

In [ ]:
# Cell 9: Flask Inference Server
# This server computes features from raw OHLCV data, then runs inference
# This fixes the feature mismatch bug in the previous implementation

from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

_model = model
_metadata = metadata
_feature_names = feature_names
_norm_stats = norm_stats


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "model": "iTransformer",
        "version": "2.0",
        "features": _metadata["num_features"],
        "lookback": LOOKBACK_WINDOW,
        "horizon": FORECAST_HORIZON,
        "architecture": _metadata["architecture"],
        "test_metrics": _metadata["test_metrics"],
        "gpu": torch.cuda.is_available(),
        "device": str(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "cpu",
    })


@app.route("/predict", methods=["POST"])
def predict():
    """
    Run prediction on OHLCV data.
    
    The server computes features from raw OHLCV, normalizes them,
    and runs the iTransformer model.

    Expects JSON:
    {
        "symbol": "AAPL",
        "current_price": 150.0,
        "features": [[open, high, low, close, volume], ...]  // 60+ days of OHLCV
    }
    """
    if _model is None:
        return jsonify({"error": "Model not loaded"}), 500

    try:
        data = request.get_json()
        symbol = data.get("symbol", "UNKNOWN")
        current_price = float(data.get("current_price", 100))
        raw_ohlcv = data["features"]  # (N, 5) — OHLCV from the API

        # Convert raw OHLCV to DataFrame
        ohlcv_df = pd.DataFrame(
            raw_ohlcv,
            columns=["Open", "High", "Low", "Close", "Volume"]
        )
        # Create a date index for feature computation
        ohlcv_df.index = pd.date_range(
            end=pd.Timestamp.now().normalize(),
            periods=len(ohlcv_df),
            freq="B"  # Business days
        )

        # Need enough data for feature computation (200-day SMA needs 200+ days)
        # But the API sends only 60 days — we compute what we can
        if len(ohlcv_df) < LOOKBACK_WINDOW:
            return jsonify({
                "error": f"Need at least {LOOKBACK_WINDOW} days of data, got {len(ohlcv_df)}"
            }), 400

        # Compute features from OHLCV (same function used in training)
        features = compute_features(ohlcv_df)

        # Ensure we have the right feature columns (in same order as training)
        feat_values = features[_feature_names].values if all(f in features.columns for f in _feature_names) else features.values

        # Normalize using stored stats (use aggregate mean across all stocks if stock-specific not available)
        if symbol in _norm_stats:
            stats = _norm_stats[symbol]
        else:
            # Use mean of all stock stats
            all_means = np.array([s["mean"] for s in _norm_stats.values()])
            all_stds = np.array([s["std"] for s in _norm_stats.values()])
            stats = {
                "mean": np.mean(all_means, axis=0).tolist(),
                "std": np.mean(all_stds, axis=0).tolist(),
            }

        feat_mean = np.array(stats["mean"]).reshape(1, -1)
        feat_std = np.array(stats["std"]).reshape(1, -1) + 1e-10

        # Handle feature count mismatch (API may send fewer features)
        n_model_features = len(_feature_names)
        n_data_features = feat_values.shape[1]
        if n_data_features < n_model_features:
            # Pad with zeros for missing features
            padding = np.zeros((feat_values.shape[0], n_model_features - n_data_features))
            feat_values = np.hstack([feat_values, padding])
            feat_mean = np.hstack([feat_mean, np.zeros((1, n_model_features - n_data_features))])
            feat_std = np.hstack([feat_std, np.ones((1, n_model_features - n_data_features))])
        elif n_data_features > n_model_features:
            feat_values = feat_values[:, :n_model_features]

        feat_norm = (feat_values - feat_mean) / feat_std
        feat_norm = np.clip(feat_norm, -5, 5)

        # Take last LOOKBACK_WINDOW days
        if feat_norm.shape[0] >= LOOKBACK_WINDOW:
            input_data = feat_norm[-LOOKBACK_WINDOW:]
        else:
            # Pad with zeros at the beginning if not enough data
            pad_size = LOOKBACK_WINDOW - feat_norm.shape[0]
            input_data = np.vstack([
                np.zeros((pad_size, feat_norm.shape[1])),
                feat_norm
            ])

        # Run inference
        input_tensor = torch.FloatTensor(input_data).unsqueeze(0).to(device)  # (1, L, V)
        with torch.no_grad():
            forecast = _model(input_tensor).cpu().numpy()[0]  # (horizon,)

        # Convert returns to prices
        predicted_prices = (current_price * (1 + forecast)).tolist()

        # Confidence bands using model prediction volatility + historical vol
        vol_20d_idx = _feature_names.index("volatility_20d") if "volatility_20d" in _feature_names else -1
        if vol_20d_idx >= 0 and vol_20d_idx < feat_values.shape[1]:
            raw_vol = feat_values[-1, vol_20d_idx]
            vol = max(0.05, min(1.0, abs(raw_vol)))  # Clamp to reasonable range
        else:
            vol = 0.2

        daily_vol = vol / np.sqrt(252)
        days = np.arange(1, FORECAST_HORIZON + 1)
        diffusion = daily_vol * np.sqrt(days)

        # 30-day prediction summary
        prediction_30d = float(forecast[-1]) * 100  # As percentage

        response = {
            "symbol": symbol,
            "model": "iTransformer",
            "model_version": "2.0",
            "forecast_returns": forecast.tolist(),
            "predicted_prices": predicted_prices,
            "prediction_30d": prediction_30d,
            "current_price": current_price,
            "horizon_days": FORECAST_HORIZON,
            "confidence": {
                "lower_95": (current_price * (1 + forecast - 1.96 * diffusion)).tolist(),
                "upper_95": (current_price * (1 + forecast + 1.96 * diffusion)).tolist(),
                "lower_68": (current_price * (1 + forecast - diffusion)).tolist(),
                "upper_68": (current_price * (1 + forecast + diffusion)).tolist(),
            },
            "model_confidence": float(max(0, min(1.0,
                _metadata["test_metrics"]["directional_accuracy_30d"] / 100
            ))),
            "metadata": {
                "num_features": _metadata["num_features"],
                "lookback": LOOKBACK_WINDOW,
                "architecture": _metadata["architecture"],
                "test_directional_accuracy": _metadata["test_metrics"]["directional_accuracy_30d"],
                "validation_method": "walk-forward",
                "loss_function": "Huber",
                "gpu_used": torch.cuda.is_available(),
            },
        }

        return jsonify(response)

    except Exception as e:
        import traceback
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500


@app.route("/feature-names", methods=["GET"])
def get_feature_names():
    return jsonify({
        "features": _feature_names or [],
        "count": len(_feature_names) if _feature_names else 0,
    })


print("[4/6] Inference server defined.")

In [ ]:
# Cell 10: Start Server with ngrok

def start_server():
    """Start Flask server with ngrok tunnel."""
    from pyngrok import ngrok

    if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "YOUR_NGROK_TOKEN_HERE":
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    else:
        print("\n[WARNING] No ngrok auth token set!")
        print("  Get your free token from: https://dashboard.ngrok.com/get-started/your-authtoken")
        print("  Set it in NGROK_AUTH_TOKEN variable above.")
        print("  Without it, the tunnel will expire quickly.\n")

    port = 5000
    public_url = ngrok.connect(port)

    print("\n" + "=" * 60)
    print("PutStrike iTransformer Inference Server v2.0")
    print("=" * 60)
    print(f"\n  Public URL:  {public_url}")
    print(f"\n  Paste this URL into PutStrike > GPU Inference (Colab) settings")
    print(f"\n  Endpoints:")
    print(f"    GET  {public_url}/health")
    print(f"    POST {public_url}/predict")
    print(f"    GET  {public_url}/feature-names")
    print(f"\n  Model: iTransformer ({_metadata['num_features']} features, {_metadata['architecture']['parameters']:,} params)")
    print(f"  Test Dir Acc: {_metadata['test_metrics']['directional_accuracy_30d']:.1f}%")
    print("=" * 60)

    threading.Thread(
        target=lambda: app.run(port=port, use_reloader=False),
        daemon=True,
    ).start()

    return str(public_url)


print("[5/6] Starting inference server...")
public_url = start_server()
print("\n[6/6] Ready! Server is running. Keep this notebook open.")